# Which song will go viral: using kNN algorithm

## Dataset description

This dataset uses different data cathegories, such as artist, genre, danceability, instrumentalness and duration to determine whether or not the song will go viral. 

In [ ]:
%pip install scikit-learn

In [ ]:
import pandas as pd
import numpy as np
from sklearn import datasets

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

In [ ]:
lets_clean_df = pd.read_csv('viral_songs.csv',sep=';')

In [ ]:
lets_clean_df

### Now to Remove "Artist_" from the values of 'artist' :

Source of Info:
(https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.str.replace)

In [ ]:
lets_clean_df['artist'] = lets_clean_df['artist'].str.replace('Artist_', '', regex=False)

In [ ]:
lets_clean_df

### Now to make the values of column loudness_db from negative to positive : 
(https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.abs)

In [ ]:
lets_clean_df['loudness_db'] = lets_clean_df['loudness_db'].abs()

In [ ]:
lets_clean_df

In [ ]:
X = lets_clean_df[['release_year', 'duration_sec', 'loudness_db']]

y = lets_clean_df['viral']

In [ ]:
X.head()

In [ ]:
y.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
print(X_train.shape)
print(X_test.shape)

In [ ]:
knn = KNeighborsClassifier(n_neighbors=3)

In [ ]:
knn.fit(X_train, y_train)

In [ ]:
y_pred = knn.predict(X_test)

In [ ]:
print(y_pred[:10])

In [ ]:
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

### Let's test with different k values.
 
When k value equals 5:

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)

knn.fit(X_train, y_train)

y_pred = knn.predict(X_test)

print(y_pred[:10])

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

When k value equals 10:

In [ ]:
knn = KNeighborsClassifier(n_neighbors=10)

knn.fit(X_train, y_train)

y_pred = knn.predict(X_test)

print(y_pred[:10])

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

## Experiment with k value findings
The bigger the k value, the bettter accuracy level, which is not something that is common in practice. Usually, with larger k value it is harder to determine the result accurately.

Overall, the accuracy score seems to be low in all cases. It is likely because of underfitting. It most likely happens, because this dataset has a lot of variables, which help to determine whether or not the song will go viral. However, in testing and learning only a few variables of data are used.

# Bhai Yaha krle apna part 

In [ ]:
from sklearn.model_selection import train_test_split, KFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
import numpy as np

## Task 5 Comparing Train/Test Splits

I compare 80/20, 70/30, and 60/40 train/test splits.
The features and number of neighbors remain unchanged.

In [ ]:
k_neighbors = 3

for test_size in [0.2, 0.3, 0.4]:

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        shuffle=True,
        random_state=42
    )

    knn = KNeighborsClassifier(n_neighbors=k_neighbors)

    knn.fit(X_train, y_train)

    y_pred = knn.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)

    print(f"Test proportion: {test_size}")
    print(f"Accuracy: {accuracy:.4f}")
    print()

### Task 5 Discussion

The classifier achieved 55.20% accuracy with an 80/20 split,
56.63% with a 70/30 split, and 57.15% with a 60/40 split.

The 60/40 split produced the highest accuracy in this experiment,
1.95 percentage points higher than the 80/20 split. Increasing
the training proportion therefore did not improve accuracy
in this run.

Changing the split ratio changes both the amount of training
data and the test examples. These results do not establish
that using less training data always improves the model.

I consider 80/20 a practical choice for this dataset because
it provides 8,000 training songs and 2,000 test songs.
However, 60/40 was the best-performing split in this particular
experiment.

Next, five fold cross validation will evaluate performance
across several different partitions to reduce reliance
on a single train/test split.

## Task 6 Five Fold Cross Validation

I use five-fold cross-validation to evaluate the classifier
on different portions of the dataset.

In [ ]:
k = 5

kf = KFold(
    n_splits=k,
    shuffle=True,
    random_state=42
)

In [ ]:
k_neighbors = 3
accuracies = []

for train_index, test_index in kf.split(X):

    X_train = X.iloc[train_index]
    X_test = X.iloc[test_index]

    y_train = y.iloc[train_index]
    y_test = y.iloc[test_index]

    knn = KNeighborsClassifier(n_neighbors=k_neighbors)

    knn.fit(X_train, y_train)

    y_pred = knn.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)

    accuracies.append(accuracy)

average_accuracy = np.mean(accuracies)

print(f"Accuracies for each fold: {accuracies}")
print(f"Average accuracy across all folds: {average_accuracy:.4f}")

### Task 6 Discussion

Five-fold cross-validation produced accuracies of 55.25%,
56.35%, 56.90%, 57.30%, and 55.70%. The average accuracy
was 56.30%.

The fold accuracies ranged from 55.25% to 57.30%, a difference
of 2.05 percentage points. This suggests relatively consistent
performance across these five partitions, although the overall
accuracy remains low.

Compared with Task 5, the cross validation average of 56.30%
was higher than the 80/20 result of 55.20%, close to the 70/30
result of 56.63%, and lower than the 60/40 result of 57.15%.

Each cross-validation round used 8,000 songs for training and
2,000 for testing. Every song was included in a test fold once.

Cross-validation provides a broader evaluation than a single
train/test split because it evaluates the model across several
partitions. The similar results across Tasks 5 and 6 suggest
that the observed performance is not limited to one particular
split. However, these results do not guarantee the same
accuracy on new songs.